# LLaVA VQA (Web 동일 로직)

이 노트북은 웹 데모(`LLaVA VQA`)와 동일한 모델/프롬프트 포맷으로, `img/xai506_example_image.jpg`를 입력으로 VQA를 실행합니다.

- Model: `llava-hf/llava-1.5-7b-hf`
- Input image: `img/xai506_example_image.jpg`
- Prompt format: `USER: <image> ... ASSISTANT:` (웹 API와 동일)


In [ ]:
# 재현 환경 자동 점검/설치
import importlib.util
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
req_file = repo_root / 'requirements_api.txt'

required_modules = {
    'torch': 'torch',
    'transformers': 'transformers',
    'PIL': 'pillow',
    'huggingface_hub': 'huggingface_hub',
}
missing = [pip_name for mod, pip_name in required_modules.items() if importlib.util.find_spec(mod) is None]

if missing:
    if req_file.exists():
        print('Installing from requirements_api.txt ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)])
    else:
        print('Installing minimal dependencies ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'transformers', 'pillow', 'huggingface_hub', 'accelerate'])
else:
    print('Dependencies already satisfied.')


In [ ]:
import torch
from pathlib import Path
from PIL import Image
from IPython.display import display
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'
IMAGE_REL_PATH = Path('img') / 'xai506_example_image.jpg'

def pick_best_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device('cpu')

    best_idx = 0
    best_free = -1
    for idx in range(torch.cuda.device_count()):
        try:
            free_bytes, _ = torch.cuda.mem_get_info(idx)
        except Exception:
            free_bytes = 0
        if free_bytes > best_free:
            best_free = free_bytes
            best_idx = idx
    return torch.device(f'cuda:{best_idx}')

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / IMAGE_REL_PATH).exists():
            return p
    raise FileNotFoundError(f'Cannot find {IMAGE_REL_PATH} from {start}')

REPO_ROOT = find_repo_root(Path.cwd())
IMAGE_PATH = REPO_ROOT / IMAGE_REL_PATH

device = pick_best_device()
dtype = torch.float16 if device.type == 'cuda' else torch.float32

print(f'Repo root: {REPO_ROOT}')
print(f'Image path: {IMAGE_PATH}')
print(f'Device: {device}, dtype: {dtype}')

image = Image.open(IMAGE_PATH).convert('RGB')
display(image)


In [ ]:
# 웹 API(get_llava_model)와 동일 구성
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation='eager',
).to(device)
model.eval()
print('Model loaded.')


In [ ]:
@torch.no_grad()
def run_llava_vqa(image: Image.Image, question: str, max_new_tokens: int = 128, temperature: float = 0.2):
    # 웹 구현(api_server.py/run_llava_vqa)와 동일한 prompt
    prompt = f'USER: <image>\n{question.strip()}\nASSISTANT:'

    inputs = processor(images=image, text=prompt, return_tensors='pt')

    model_inputs = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            if k == 'pixel_values':
                model_inputs[k] = v.to(device=device, dtype=dtype)
            else:
                model_inputs[k] = v.to(device=device)
        else:
            model_inputs[k] = v

    do_sample = temperature > 0
    generation_kwargs = {
        'max_new_tokens': max(1, int(max_new_tokens)),
        'do_sample': do_sample,
    }
    if do_sample:
        generation_kwargs['temperature'] = max(float(temperature), 1e-4)

    generation = model.generate(**model_inputs, **generation_kwargs)

    prompt_len = model_inputs['input_ids'].shape[1]
    answer_tokens = generation[:, prompt_len:]

    if answer_tokens.shape[1] == 0:
        decoded = processor.batch_decode(generation, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        answer = decoded.replace(prompt, '').strip()
    else:
        answer = processor.batch_decode(
            answer_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0].strip()

    return {
        'answer': answer,
        'prompt': prompt,
        'model_id': MODEL_ID,
        'device': str(device),
        'dtype': str(dtype).replace('torch.', ''),
    }


In [ ]:
# 질문만 바꿔서 재실행하세요
question = 'What is happening in this image? Please answer in Korean.'
max_new_tokens = 128
temperature = 0.2

result = run_llava_vqa(
    image=image,
    question=question,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
)

print('Model:', result['model_id'])
print('Device:', result['device'])
print('DType:', result['dtype'])
print('--- Question ---')
print(question)
print('--- Answer ---')
print(result['answer'])


## 재현 팁
- GPU가 있으면 자동으로 `cuda` + `float16`을 사용합니다.
- 결과 일관성을 높이려면 `temperature=0`으로 설정하세요.
- 현재 웹 데모와 동일하게 `temperature=0.2` 기본값을 사용했습니다.
